In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from skfuzzy import control as ctrl
import skfuzzy as fuzz
import pickle

import random
import copy


In [20]:
file_path = r"..\data\processed\healthcare_processed.csv"
fuzzy_data = pd.read_csv(file_path)
# fuzzy_data.head()

# Correct feature set for fuzzy-GA system
X_fuzzy = fuzzy_data[['age', 'bmi','gender','ever_married',
                      'work_type','smoking_status','avg_glucose_level',
                      'hypertension', 'heart_disease', 'Residence_type']]

y_fuzzy = fuzzy_data['stroke_risk']  # target classes: 0=Low, 1=Medium, 2=High

print("X shape:", X_fuzzy.shape)
print("y shape:", y_fuzzy.shape)


X shape: (5110, 10)
y shape: (5110,)


In [21]:


# Notebook 4 — load spec and rebuild fuzzy system

with open(r"..\\data\\processed\\fuzzy_system_spec.pkl", "rb") as f:
    spec = pickle.load(f)

# Helper to create antecedent/consequent objects and add MFs
def create_var(name, universe_min, universe_max, step=1):
    uni = np.arange(universe_min, universe_max+step, step)
    var = ctrl.Antecedent(uni, name)
    return var

# Create variables
vars_ = {}
for vname, (lo, hi) in spec['universes'].items():
    vars_[vname] = create_var(vname, lo, hi, step=1)

# Add MFs from spec to each var
def add_mf(var_obj, mf_name, mf_spec):
    if mf_spec['type'] == 'trimf':
        params = mf_spec['params']
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, params)
    elif mf_spec['type'] == 'trap':
        params = mf_spec['params']
        var_obj[mf_name] = fuzz.trapmf(var_obj.universe, params)
    elif mf_spec['type'] == 'singleton':
        p = mf_spec['params'][0]
        eps = 0.001 * (var_obj.universe.max() - var_obj.universe.min())
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, [p-eps, p, p+eps])
    else:
        raise ValueError("Unknown MF type: "+mf_spec['type'])

# Apply MFs from spec
for vname, mfs in spec['mfs'].items():
    for mf_name, mf_spec in mfs.items():
        add_mf(vars_[vname], mf_name, mf_spec)

# Recreate rules
rules_list = []
for antecedent, consequent in spec['rules']:
    left = None
    for feat, mf_name in antecedent.items():
        cond = vars_[feat][mf_name]
        if left is None:
            left = cond
        else:
            left = left & cond

    right_feat, right_mf = list(consequent.items())[0]

    if 'stroke_risk' not in vars_:
        out = ctrl.Consequent(np.arange(0,3,1), 'stroke_risk')
        out['low'] = fuzz.trimf(out.universe, [0,0,1])
        out['medium'] = fuzz.trimf(out.universe, [0,1,2])
        out['high'] = fuzz.trimf(out.universe, [1,2,2])
        vars_['stroke_risk'] = out

    right = vars_['stroke_risk'][right_mf]
    rules_list.append(ctrl.Rule(left, right))

stroke_ctrl = ctrl.ControlSystem(rules_list)
stroke_sim = ctrl.ControlSystemSimulation(stroke_ctrl)

print("Rebuilt fuzzy system from spec. Rules:", len(rules_list))


Rebuilt fuzzy system from spec. Rules: 18


In [22]:
# ---------- Use spec (from NB3) to build chromosome & encode rules ----------



# Sanity: ensure 'spec' exists
try:
    spec
except NameError:
    raise RuntimeError("spec not found — make sure you loaded fuzzy_system_spec.pkl as 'spec' earlier.")

# 1) Define which features GA should optimize (numeric continuous)
numeric_features = ['age', 'bmi', 'avg_glucose_level']

# 2) helper: map real param -> normalized [0,1] relative to universe
def normalize_param(val, lo, hi):
    if hi == lo:
        return 0.0
    return (val - lo) / (hi - lo)

def denormalize_param(norm, lo, hi):
    return lo + norm * (hi - lo)

# 3) Build chromosome initial vector from spec (normalize numeric-MF params)
chromosome_parts = []
numeric_mf_meta = {}

for feat in numeric_features:
    uni_lo, uni_hi = spec['universes'][feat][0], spec['universes'][feat][1]
    mfs = spec['mfs'][feat]

    preferred_order = ['young','low','normal','middle','elevated','high','old']
    mf_names = [name for name in preferred_order if name in mfs]

    if not mf_names:
        mf_names = sorted(mfs.keys())

    numeric_mf_meta[feat] = []
    for mf_name in mf_names:
        mf_spec = mfs[mf_name]
        params = mf_spec['params']

        norm_params = [normalize_param(p, uni_lo, uni_hi) for p in params]
        chromosome_parts.extend(norm_params)

        numeric_mf_meta[feat].append({
            'mf_name': mf_name,
            'type': mf_spec['type'],
            'params_original': params,
            'uni_lo': uni_lo,
            'uni_hi': uni_hi,
            'n_params': len(params)
        })

initial_chromosome = np.array(chromosome_parts, dtype=float)
print("Initial chromosome length (numeric-only):", initial_chromosome.size)

# Build index map
index_map = {}
idx = 0
for feat, metas in numeric_mf_meta.items():
    n_params = sum(m['n_params'] for m in metas)
    index_map[feat] = (idx, n_params)
    idx += n_params

print("Index map per feature:", index_map)

# 4) Encode rule templates
rule_templates = []
for antecedent, consequent in spec['rules']:
    rule_templates.append({'antecedent': antecedent, 'consequent': consequent})

print("Number of rule templates loaded:", len(rule_templates))
for r in rule_templates[:5]:
    print(r)

# 5) Decode helper
def decode_numeric_chromosome(chromosome):
    decoded = {}
    for feat, (start, n) in index_map.items():
        seg = chromosome[start:start+n]
        metas = numeric_mf_meta[feat]
        out = []
        j = 0
        for meta in metas:
            np_count = meta['n_params']
            seg_part = seg[j:j+np_count]
            real_params = [denormalize_param(val, meta['uni_lo'], meta['uni_hi']) for val in seg_part]
            out.append({'mf_name': meta['mf_name'], 'type': meta['type'], 'params': real_params})
            j += np_count
        decoded[feat] = out
    return decoded

# Test decode
decoded_test = decode_numeric_chromosome(initial_chromosome)
for feat, mfs in decoded_test.items():
    print(f"\n{feat}:")
    for m in mfs:
        print(" ", m['mf_name'], m['type'], m['params'])

# Save meta + chromosome
with open(r"..\\data\\processed\\nb4_chromosome_meta.pkl", "wb") as f:
    pickle.dump({
        'initial_chromosome': initial_chromosome,
        'numeric_mf_meta': numeric_mf_meta,
        'index_map': index_map,
        'rule_templates': rule_templates,
        'spec_universes': spec['universes']
    }, f)

print("Saved nb4_chromosome_meta.pkl with initial chromosome and meta.")


Initial chromosome length (numeric-only): 33
Index map per feature: {'age': (0, 11), 'bmi': (11, 11), 'avg_glucose_level': (22, 11)}
Number of rule templates loaded: 18
{'antecedent': {'age': 'old', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'high'}}
{'antecedent': {'age': 'old', 'heart_disease': 'yes'}, 'consequent': {'stroke_risk': 'high'}}
{'antecedent': {'age': 'middle', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'medium'}}
{'antecedent': {'age': 'young', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'medium'}}
{'antecedent': {'avg_glucose_level': 'high', 'bmi': 'high'}, 'consequent': {'stroke_risk': 'high'}}

age:
  young trap [0.0, 0.0, 25.0, 35.0]
  middle trimf [30.0, 45.0, 60.0]
  old trap [55.00000000000001, 65.0, 100.0, 100.0]

bmi:
  low trap [10.0, 10.0, 18.0, 22.0]
  normal trimf [20.0, 25.0, 30.0]
  high trap [28.0, 32.0, 60.0, 60.0]

avg_glucose_level:
  normal trap [50.0, 50.0, 90.0, 110.0]
  elevated trimf [100.0, 130.0, 160.0]
  high tra

In [23]:
# ------------------------------------------------------------
# BUILD FUZZY SYSTEM FROM CHROMOSOME — FIXED & FULLY COMPATIBLE
# ------------------------------------------------------------

import skfuzzy as fuzz
from skfuzzy import control as ctrl


def build_fuzzy_system_from_chromosome(
    chrom,
    universes,
    numeric_mf_meta,
    index_map,
    rule_templates
):
    """
    Rebuild fuzzy system using:
    - chrom: numeric GA chromosome
    - universes: spec['universes']
    - numeric_mf_meta: meta for numeric MF structure
    - index_map: mapping chromosome segments
    - rule_templates: list of {antecedent, consequent}
    """

    fuzzy_vars = {}

    # ---------------------------------------------------
    # 1) CREATE INPUT VARIABLES + ASSIGN MFs
    # ---------------------------------------------------
    for var_name, (lo, hi) in universes.items():

        # Create antecedent (input)
        if var_name != 'stroke_risk':
            fuzzy_vars[var_name] = ctrl.Antecedent(
                np.arange(lo, hi + 1, 1), var_name
            )
        else:
            # Output variable
            fuzzy_vars[var_name] = ctrl.Consequent(
                np.arange(lo, hi + 1, 1), var_name
            )

    # ---------------------------------------------------
    # 2) ASSIGN MEMBERSHIP FUNCTIONS
    # ---------------------------------------------------

    for feat, metas in numeric_mf_meta.items():

        start_idx, _ = index_map[feat]
        idx = start_idx

        for meta in metas:
            mf_name = meta['mf_name']
            mf_type = meta['type']
            n_params = meta['n_params']
            lo = meta['uni_lo']
            hi = meta['uni_hi']

            # extract parameters from chromosome
            segment = chrom[idx:idx + n_params]
            idx += n_params

            # denormalize parameters
            params = [lo + p * (hi - lo) for p in segment]

            # assign MF
            if mf_type == 'trimf':
                fuzzy_vars[feat][mf_name] = fuzz.trimf(
                    fuzzy_vars[feat].universe, params
                )
            elif mf_type in ['trap', 'trapmf']:
                fuzzy_vars[feat][mf_name] = fuzz.trapmf(
                    fuzzy_vars[feat].universe, params
                )

    # ---------------------------------------------------
    # 3) BUILD RULES
    # ---------------------------------------------------

    rules = []
    for rule in rule_templates:

        left_expr = None

        for feat, mf_name in rule['antecedent'].items():
            cond = fuzzy_vars[feat][mf_name]
            left_expr = cond if left_expr is None else left_expr & cond

        out_feat, out_mf = list(rule['consequent'].items())[0]
        right_expr = fuzzy_vars[out_feat][out_mf]

        rules.append(ctrl.Rule(left_expr, right_expr))

    # ---------------------------------------------------
    # 4) CONTROL SYSTEM
    # ---------------------------------------------------
    system = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(system)

    return fuzzy_vars, rules, system, sim


In [24]:
def fitness_function(chromosome, fuzzy_vars_spec):
    """
    Build fuzzy system from chromosome,
    run simulation on all samples,
    compute accuracy.
    """

    import skfuzzy.control as ctrl

    # 1) Build fuzzy system properly
    fuzzy_vars, rules, system, sim_template = build_fuzzy_system_from_chromosome(
        chromosome,
        fuzzy_vars_spec
    )

    preds = []

    # 2) Loop through dataset
    for i in range(len(X_fuzzy)):

        # Create a new simulation object each time
        sim = ctrl.ControlSystemSimulation(system)

        # assign all fuzzy inputs
        for var in X_fuzzy.columns:
            sim.input[var] = float(X_fuzzy.iloc[i][var])

        # compute output
        try:
            sim.compute()
            pred_value = sim.output['stroke_risk']
        except:
            pred_value = 1  # fallback (medium risk)

        # convert to class: 0/1/2
        preds.append(int(round(pred_value)))

    # 3) compute accuracy
    acc = accuracy_score(y_fuzzy, preds)
    return acc


In [25]:
# ---------------------------
# ADVANCED GENETIC ALGORITHM (FIXED)
# ---------------------------
import numpy as np
import random
from joblib import Parallel, delayed

# GA Hyperparameters (tweak as needed)
POP_SIZE = 40
GENERATIONS = 50
TOURNAMENT_K = 4
ELITE_COUNT = 3

INITIAL_MUTATION = 0.25
MIN_MUTATION = 0.05
MAX_MUTATION = 0.4

CROSSOVER_RATE = 0.85
ALPHA = 0.4   # BLX-alpha parameter

# ---------------------------
# Helper: Tournament selection
# ---------------------------
def tournament_selection(population, fitness_scores, k=TOURNAMENT_K):
    idxs = random.sample(range(len(population)), k)
    best_idx = max(idxs, key=lambda i: fitness_scores[i])
    # return a copy to avoid accidental in-place modification
    return population[best_idx].copy()

# ---------------------------
# Helper: BLX-alpha crossover (bounded to [0,1])
# ---------------------------
def blx_alpha_crossover(parent1, parent2, alpha=ALPHA):
    # parents expected as numpy arrays
    p1 = parent1.copy()
    p2 = parent2.copy()
    c1 = p1.copy()
    c2 = p2.copy()
    for i in range(len(p1)):
        x1 = p1[i]
        x2 = p2[i]
        low = min(x1, x2)
        high = max(x1, x2)
        diff = high - low
        cmin = low - alpha * diff
        cmax = high + alpha * diff
        c1[i] = random.uniform(cmin, cmax)
        c2[i] = random.uniform(cmin, cmax)
    # ensure valid bounds
    c1 = np.clip(c1, 0.0, 1.0)
    c2 = np.clip(c2, 0.0, 1.0)
    return c1, c2

# ---------------------------
# Helper: Adaptive mutation (Gaussian) with clipping
# ---------------------------
def adaptive_mutation(chromosome, mutation_rate):
    child = chromosome.copy()
    # sigma scales with mutation_rate (keeps mutations meaningful but bounded)
    sigma = 0.08 + 0.3 * mutation_rate
    mask = np.random.rand(child.size) < mutation_rate
    # add gaussian noise only where mask True
    noise = np.random.normal(0, sigma, size=child.shape)
    child[mask] = child[mask] + noise[mask]
    # clip to [0,1]
    child = np.clip(child, 0.0, 1.0)
    return child

# ---------------------------
# Penalize bad MFs (light penalty)
# ---------------------------
def penalty_function(chromosome):
    penalty = 0.0
    # prefer using index_map variable (assumed available)
    for feature_name, (start, length) in index_map.items():
        vals = chromosome[start:start+length]
        # small penalty if any value out of bounds (shouldn't happen due to clipping)
        if np.any(vals < 0) or np.any(vals > 1):
            penalty += 0.02
        # penalty for unordered parameters within each MF (small)
        # we need to iterate according to numeric_mf_meta for this feature
        metas = numeric_mf_meta.get(feature_name, [])
        pos = 0
        for meta in metas:
            n = meta['n_params']
            segment = vals[pos:pos+n]
            pos += n
            # check non-decreasing within a membership function
            for j in range(len(segment)-1):
                if segment[j] > segment[j+1] + 1e-8:
                    penalty += 0.01
    return penalty

# ---------------------------
# FITNESS wrapper with penalty (uses existing fitness_function)
# ---------------------------
def fitness_with_penalty(chromosome):
    # ensure numpy array
    chrom = np.array(chromosome, dtype=float)
    base_acc = fitness_function(chrom)  # assumes fitness_function accepts numpy array
    pen = penalty_function(chrom)
    # keep fitness in [ -inf, 1 ] domain; subtract small penalty
    return base_acc - pen

# ---------------------------
# MAIN GA LOOP
# ---------------------------
def run_advanced_ga(pop_size=POP_SIZE, generations=GENERATIONS):
    # initialize population: perturb initial_chromosome slightly for diversity
    population = []
    for _ in range(pop_size):
        jitter = np.random.normal(0, 0.01, size=initial_chromosome.shape)
        p = np.clip(initial_chromosome + jitter, 0.0, 1.0)
        population.append(p)

    mutation_rate = INITIAL_MUTATION
    best_scores = []

    for gen in range(generations):
        # parallel fitness evaluation
        fitness_scores = Parallel(n_jobs=-1)(
            delayed(fitness_with_penalty)(chrom) for chrom in population
        )
        fitness_scores = np.array(fitness_scores)
        best_idx = int(np.argmax(fitness_scores))
        best_scores.append(float(fitness_scores[best_idx]))

        print(f"Generation {gen+1}/{generations} | Best fitness = {best_scores[-1]:.4f}")

        # elitism: keep top ELITE_COUNT
        elites_idx = np.argsort(fitness_scores)[-ELITE_COUNT:]
        elites = [population[i].copy() for i in elites_idx]

        # create next generation
        new_population = elites.copy()
        while len(new_population) < pop_size:
            p1 = tournament_selection(population, fitness_scores)
            p2 = tournament_selection(population, fitness_scores)

            if random.random() < CROSSOVER_RATE:
                c1, c2 = blx_alpha_crossover(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()

            c1 = adaptive_mutation(c1, mutation_rate)
            c2 = adaptive_mutation(c2, mutation_rate)

            new_population.append(c1)
            if len(new_population) < pop_size:
                new_population.append(c2)

        population = new_population

        # adapt mutation rate based on improvement (simple schedule)
        if gen > 0:
            if best_scores[-1] <= best_scores[-2]:
                mutation_rate = min(MAX_MUTATION, mutation_rate * 1.12)
            else:
                mutation_rate = max(MIN_MUTATION, mutation_rate * 0.92)

    # final evaluation & return best
    fitness_scores = Parallel(n_jobs=-1)(
        delayed(fitness_with_penalty)(chrom) for chrom in population
    )
    fitness_scores = np.array(fitness_scores)
    best_idx = int(np.argmax(fitness_scores))
    best_chromosome = population[best_idx]

    print("\nGA DONE!")
    print("Best GA fitness:", fitness_scores[best_idx])

    return best_chromosome, best_scores


In [27]:
# %%
# IMPORTS
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from skfuzzy import control as ctrl
import skfuzzy as fuzz
import pickle
import random
import copy
from joblib import Parallel, delayed

# %%
# LOAD DATA
file_path = r"..\data\processed\healthcare_processed.csv"
fuzzy_data = pd.read_csv(file_path)
X_fuzzy = fuzzy_data[['age', 'bmi','gender','ever_married','work_type','smoking_status','avg_glucose_level','hypertension', 'heart_disease', 'Residence_type']]
y_fuzzy = fuzzy_data['stroke_risk']
print("X shape:", X_fuzzy.shape)
print("y shape:", y_fuzzy.shape)

# %%
# LOAD FUZZY SYSTEM SPEC
with open(r"..\\data\\processed\\fuzzy_system_spec.pkl", "rb") as f:
    spec = pickle.load(f)

# %%
# HELPER: CREATE VARIABLES AND MFS
def create_var(name, universe_min, universe_max, step=1):
    uni = np.arange(universe_min, universe_max+step, step)
    return ctrl.Antecedent(uni, name)

def add_mf(var_obj, mf_name, mf_spec):
    if mf_spec['type'] == 'trimf':
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, mf_spec['params'])
    elif mf_spec['type'] in ['trap', 'trapmf']:
        var_obj[mf_name] = fuzz.trapmf(var_obj.universe, mf_spec['params'])
    elif mf_spec['type'] == 'singleton':
        p = mf_spec['params'][0]
        eps = 0.001 * (var_obj.universe.max() - var_obj.universe.min())
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, [p-eps, p, p+eps])
    else:
        raise ValueError("Unknown MF type: "+mf_spec['type'])

# %%
# BUILD VARIABLES
vars_ = {}
for vname, (lo, hi) in spec['universes'].items():
    vars_[vname] = create_var(vname, lo, hi)
for vname, mfs in spec['mfs'].items():
    for mf_name, mf_spec in mfs.items():
        add_mf(vars_[vname], mf_name, mf_spec)

# %%
# RECREATE RULES
rules_list = []
for antecedent, consequent in spec['rules']:
    left = None
    for feat, mf_name in antecedent.items():
        cond = vars_[feat][mf_name]
        left = cond if left is None else left & cond
    if 'stroke_risk' not in vars_:
        out = ctrl.Consequent(np.arange(0,3,1), 'stroke_risk')
        out['low'] = fuzz.trimf(out.universe, [0,0,1])
        out['medium'] = fuzz.trimf(out.universe, [0,1,2])
        out['high'] = fuzz.trimf(out.universe, [1,2,2])
        vars_['stroke_risk'] = out
    right_feat, right_mf = list(consequent.items())[0]
    right = vars_['stroke_risk'][right_mf]
    rules_list.append(ctrl.Rule(left, right))

stroke_ctrl = ctrl.ControlSystem(rules_list)
stroke_sim = ctrl.ControlSystemSimulation(stroke_ctrl)
print("Rebuilt fuzzy system from spec. Rules:", len(rules_list))

# %%
# BUILD CHROMOSOME
numeric_features = ['age', 'bmi', 'avg_glucose_level']

# NORMALIZATION HELPERS
def normalize_param(val, lo, hi):
    return 0.0 if hi==lo else (val-lo)/(hi-lo)

def denormalize_param(norm, lo, hi):
    return lo + norm*(hi-lo)

# BUILD INITIAL CHROMOSOME
chromosome_parts = []
numeric_mf_meta = {}
for feat in numeric_features:
    uni_lo, uni_hi = spec['universes'][feat]
    mfs = spec['mfs'][feat]
    mf_names = [n for n in ['young','low','normal','middle','elevated','high','old'] if n in mfs]
    if not mf_names: mf_names = sorted(mfs.keys())
    numeric_mf_meta[feat] = []
    for mf_name in mf_names:
        params = mfs[mf_name]['params']
        norm_params = [normalize_param(p, uni_lo, uni_hi) for p in params]
        chromosome_parts.extend(norm_params)
        numeric_mf_meta[feat].append({'mf_name': mf_name,'type': mfs[mf_name]['type'],'params_original': params,'uni_lo': uni_lo,'uni_hi': uni_hi,'n_params': len(params)})

initial_chromosome = np.array(chromosome_parts, dtype=float)

# INDEX MAP
index_map = {}
idx = 0
for feat, metas in numeric_mf_meta.items():
    n_params = sum(m['n_params'] for m in metas)
    index_map[feat] = (idx, n_params)
    idx += n_params

# RULE TEMPLATES
rule_templates = [{'antecedent': a,'consequent': c} for a,c in spec['rules']]

# %%
# BUILD FUZZY SYSTEM FROM CHROMOSOME (FUNCTION)
def build_fuzzy_system_from_chromosome(chrom, universes, numeric_mf_meta, index_map, rule_templates):
    fuzzy_vars = {}
    for var_name, (lo, hi) in universes.items():
        if var_name != 'stroke_risk':
            fuzzy_vars[var_name] = ctrl.Antecedent(np.arange(lo, hi+1, 1), var_name)
        else:
            fuzzy_vars[var_name] = ctrl.Consequent(np.arange(lo, hi+1, 1), var_name)
    for feat, metas in numeric_mf_meta.items():
        start_idx, _ = index_map[feat]
        idx = start_idx
        for meta in metas:
            segment = chrom[idx:idx+meta['n_params']]
            idx += meta['n_params']
            params = [meta['uni_lo'] + p*(meta['uni_hi']-meta['uni_lo']) for p in segment]
            if meta['type']=='trimf':
                fuzzy_vars[feat][meta['mf_name']] = fuzz.trimf(fuzzy_vars[feat].universe, params)
            elif meta['type'] in ['trap','trapmf']:
                fuzzy_vars[feat][meta['mf_name']] = fuzz.trapmf(fuzzy_vars[feat].universe, params)
    rules = []
    for rule in rule_templates:
        left_expr = None
        for feat, mf_name in rule['antecedent'].items():
            cond = fuzzy_vars[feat][mf_name]
            left_expr = cond if left_expr is None else left_expr & cond
        out_feat, out_mf = list(rule['consequent'].items())[0]
        right_expr = fuzzy_vars[out_feat][out_mf]
        rules.append(ctrl.Rule(left_expr, right_expr))
    system = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(system)
    return fuzzy_vars, rules, system, sim

# %%
# FITNESS FUNCTION
def fitness_function(chromosome):
    _, _, system, _ = build_fuzzy_system_from_chromosome(chromosome, spec['universes'], numeric_mf_meta, index_map, rule_templates)
    preds = []
    for i in range(len(X_fuzzy)):
        sim = ctrl.ControlSystemSimulation(system)
        for var in X_fuzzy.columns:
            sim.input[var] = float(X_fuzzy.iloc[i][var])
        try: sim.compute(); pred_value = sim.output['stroke_risk']
        except: pred_value = 1
        preds.append(int(round(pred_value)))
    return accuracy_score(y_fuzzy, preds)

# %%
# GA FUNCTION PLACEHOLDERS (tournament, crossover, mutation, penalty)
def tournament_selection(population, fitness_scores, k=4):
    idxs = random.sample(range(len(population)), k)
    return population[max(idxs, key=lambda i: fitness_scores[i])].copy()

def blx_alpha_crossover(p1, p2, alpha=0.4):
    c1, c2 = p1.copy(), p2.copy()
    for i in range(len(p1)):
        low, high = min(p1[i], p2[i]), max(p1[i], p2[i])
        diff = high - low
        c1[i] = random.uniform(low-alpha*diff, high+alpha*diff)
        c2[i] = random.uniform(low-alpha*diff, high+alpha*diff)
    return np.clip(c1,0,1), np.clip(c2,0,1)

def adaptive_mutation(chrom, rate):
    child = chrom.copy()
    sigma = 0.08 + 0.3*rate
    mask = np.random.rand(child.size) < rate
    noise = np.random.normal(0,sigma, size=child.shape)
    child[mask] += noise[mask]
    return np.clip(child,0,1)

def penalty_function(chrom):
    pen = 0.0
    for feat, (start, length) in index_map.items():
        vals = chrom[start:start+length]
        if np.any(vals<0) or np.any(vals>1): pen+=0.02
        metas = numeric_mf_meta.get(feat, [])
        pos=0
        for meta in metas:
            seg=vals[pos:pos+meta['n_params']]; pos+=meta['n_params']
            for j in range(len(seg)-1):
                if seg[j]>seg[j+1]+1e-8: pen+=0.01
    return pen

def fitness_with_penalty(chrom):
    return fitness_function(chrom)-penalty_function(chrom)

# %%
# MAIN GA LOOP
def run_advanced_ga():
    population = [np.clip(initial_chromosome + np.random.normal(0,0.01,size=initial_chromosome.shape),0,1) for _ in range(40)]
    best_scores=[]; mutation_rate=0.25
    for gen in range(50):
        fitness_scores = np.array(Parallel(n_jobs=-1)(delayed(fitness_with_penalty)(c) for c in population))
        best_idx = np.argmax(fitness_scores); best_scores.append(fitness_scores[best_idx])
        print(f"Generation {gen+1}/50 | Best fitness = {best_scores[-1]:.4f}")
        elites = [population[i].copy() for i in np.argsort(fitness_scores)[-3:]]
        new_population = elites.copy()
        while len(new_population)<len(population):
            p1 = tournament_selection(population, fitness_scores)
            p2 = tournament_selection(population, fitness_scores)
            if random.random()<0.85: c1,c2 = blx_alpha_crossover(p1,p2)
            else: c1,c2=p1.copy(),p2.copy()
            c1=adaptive_mutation(c1, mutation_rate); c2=adaptive_mutation(c2, mutation_rate)
            new_population.append(c1)
            if len(new_population)<len(population): new_population.append(c2)
        population=new_population
        if gen>0: mutation_rate = min(0.4, mutation_rate*1.12) if best_scores[-1]<=best_scores[-2] else max(0.05, mutation_rate*0.92)
    fitness_scores = np.array(Parallel(n_jobs=-1)(delayed(fitness_with_penalty)(c) for c in population))
    best_chrom = population[np.argmax(fitness_scores)]
    print("\nGA DONE! Best GA fitness:", max(fitness_scores))
    return best_chrom, best_scores


X shape: (5110, 10)
y shape: (5110,)
Rebuilt fuzzy system from spec. Rules: 18
